# BizLens v2.2.12 - Linear & Multiple Linear Regression (Fully Enriched)

Complete regression analysis with **regression equation display**, scatter plots, correlation heatmap, and sub-plots.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/solutiongate-learn/bizlens.git"])
print("✅ BizLens v2.2.12 ready!")

In [ ]:
import bizlens as bl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, mean_absolute_error

sns.set_style("whitegrid")
# bl.set_profiling(True)

## 1. Load Dataset

In [ ]:
df = bl.load_dataset("diamonds")
bl.describe(df)

## 2. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 3. Simple Linear Regression (Price ~ Carat)

In [ ]:
X_simple = sm.add_constant(df['carat'])
y = df['price']
model_simple = sm.OLS(y, X_simple).fit()

# Display regression equation
print(f"Regression Equation: Price = {model_simple.params[0]:.2f} + {model_simple.params[1]:.2f} × Carat")
print(model_simple.summary())

In [ ]:
# Scatter plot with fitted line
plt.scatter(df['carat'], df['price'], alpha=0.6)
plt.plot(df['carat'], model_simple.predict(X_simple), color='red', linewidth=2)
plt.title('Simple Linear Regression: Price vs Carat')
plt.xlabel('Carat')
plt.ylabel('Price')
plt.show()

## 4. Multiple Linear Regression + Sub-plots

In [ ]:
df_reg = pd.get_dummies(df[['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'price']], drop_first=True)
X = sm.add_constant(df_reg.drop('price', axis=1))
y = df_reg['price']

model_multi = sm.OLS(y, X).fit()

# Display full regression equation
equation = "Price = " + f"{model_multi.params[0]:.2f}"
for i, coef in enumerate(model_multi.params[1:]):
    equation += f" + {coef:.2f} × {X.columns[i+1]}"
print("Multiple Regression Equation:")
print(equation)

print(model_multi.summary())

## 5. Pairplot & Individual Sub-plots

In [ ]:
# Pairplot for key variables
sns.pairplot(df[['carat', 'depth', 'table', 'price']], diag_kind='kde')
plt.show()

In [ ]:
# Individual scatter plots for top predictors
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
variables = ['carat', 'depth', 'table']

for i, var in enumerate(variables):
    sns.regplot(x=var, y='price', data=df, ax=axes[i], scatter_kws={'alpha':0.6}, line_kws={'color':'red'})
    axes[i].set_title(f'Price vs {var}')

plt.tight_layout()
plt.show()

## 6. Model Metrics & Residual Analysis

In [ ]:
fitted = model_multi.fittedvalues
residuals = model_multi.resid

sst = np.sum((y - y.mean())**2)
ssr = np.sum((fitted - y.mean())**2)
sse = np.sum(residuals**2)

print(f"SST: {sst:,.2f} | SSR: {ssr:,.2f} | SSE: {sse:,.2f}")
print(f"R² = {model_multi.rsquared:.4f} | Adjusted R² = {model_multi.rsquared_adj:.4f}")
print(f"MSE = {mean_squared_error(y, fitted):,.2f} | RMSE = {np.sqrt(mean_squared_error(y, fitted)):,.2f}")